### Phase1: Build a Schema
Given a JSON document, figure out which fields are searchable and what type they are.

(hardcode now, automate later)

In [1]:
from datetime import datetime

In [2]:
def discover_fields(doc):
  fields=[]
  for key in doc:
      fields.append(key)
  return fields

In [6]:
def determine_type(value):
  if(isinstance(value,int)):
    return "int"
  elif(isinstance(value,float)):
    return "float"
  elif(isinstance(value,bool)):
    return "bool"
  elif(isinstance(value,str)):
    try:
      datetime.strptime(value,"%Y-%m-%d")
      return "date"
    except ValueError:
      return "categorical"

In [9]:
def build_schema(doc):
  schema={}
  for key,value in doc.items():
    field_type=determine_type(value)
    schema[key] = {
            "type": field_type,
            "indexed": field_type=="categorical"
        }
  return schema

In [10]:
doc = {
    "profile.firstName": "Alex",
    "profile.lastName": "Chen",
    "profile.dateOfBirth": "1995-08-14",
    "metadata.loginCount": 342,
    "preferences.theme": "dark"
}
schema=build_schema(doc)
print(schema)

{'profile.firstName': {'type': 'categorical', 'indexed': True}, 'profile.lastName': {'type': 'categorical', 'indexed': True}, 'profile.dateOfBirth': {'type': 'date', 'indexed': False}, 'metadata.loginCount': {'type': 'int', 'indexed': False}, 'preferences.theme': {'type': 'categorical', 'indexed': True}}


### Phase 2: Single ingest

In [ ]:
def index_document(doc,doc_id,schema,index):
  for field,value in doc.items():
    if not schema[field]["indexed"]:
      continue
    if field not in index:
      index[field]={}
    if value not in index[field]:
      index[field][value]=set()
    index[field][value].add(doc_id)

In [18]:
doc1 = {
    "profile.firstName": "Alex",
    "preferences.theme": "dark",
    "metadata.loginCount": 342
}

doc2 = {
    "profile.firstName": "John",
    "preferences.theme": "dark",
    "metadata.loginCount": 100
}

doc3 = {
    "profile.firstName": "Alex",
    "preferences.theme": "light",
    "metadata.loginCount": 200
}

In [19]:
schema = build_schema(doc1)
index = {}
index_document(doc1, 0, schema, index)
print(index)

{'profile.firstName': {'Alex': {0}}, 'preferences.theme': {'dark': {0}}}


In [ ]:
index_document(doc2, 1, schema, index)

In [21]:
index_document(doc3, 2, schema, index)
print(index)

{'profile.firstName': {'Alex': {0, 2}, 'John': {1}}, 'preferences.theme': {'dark': {0, 1}, 'light': {2}}}


#### query engine

In [23]:
def query_equals(field,value,index):
  if field not in index:
    return set()
  if value not in index[field]:
    return set()
  return index[field][value]

In [24]:
print(query_equals("profile.firstName","Alex",index))

{0, 2}


In [25]:
# eg, results = [
#     query_equals("profile.firstName", "Alex", index),
#     query_equals("preferences.theme", "dark", index),
#     query_equals("country", "USA", index)
# ]
def query_and(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer=answer & q
  return answer

In [26]:
def query_or(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer|=q
  return answer

### Simple BitMap implementation

In [2]:
class BitMap:
  def __init__(self):
    self.bits=0
  # Initially,everything absent:0000000000000000

  # Add an element
  def add(self,x):
    self.bits |=(1<<x)
  # to check if an element exists
  def contains(self,x):
    return (self.bits & (1<<x))!=0
  # remove an element
  def remove(self,x):
    self.bits&= ~(1<<x)
  def __str__(self):
    return bin(self.bits)

In [3]:
bm=BitMap()
bm.add(2)
bm.add(5)
print(bm)

0b100100


In [4]:
bm.add(1)
bm.add(4)
bm.add(7)

In [5]:
print(bm)

0b10110110


In [6]:
print(bm.contains(4))

True


In [7]:
bm.remove(4)

In [8]:
print(bm.contains(4))

False


In [10]:
print(bm)

0b10100110


#### Next Improvement:
In languages like C++: unsigned long long has only 64 bits, So you can't even store bit 1000.
Instead of one gigantic integer, we'll store many 64-bit integers.

then

word = x // 64

bit = x % 64